# XGBoost - Cross Validation (WDBC)

Notebook para executar XGBoost com StratifiedKFold sobre o dataset WDBC.
- Selecione qual dataset normalizado usar alterando a variável `DATASET_PATH` no bloco de configuração.
- Os parâmetros do modelo estão em `xgb_params`.
- Outputs: `model_reports/xgboost/cv/` contém `csv/`, `images/`, `pdf/`.
- O modelo será salvo em `databases/WDBC/common/models/xgboost/v1/xgb_model_cf{N_SPLITS}.pkl`.

In [1]:
# Configuração
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = Path('../../')  # ajustável se necessário
# Escolha um dos 3 datasets normalizados gerados pelo EDA: 'Standard', 'Robust', 'MinMax'
DATASET_NAME = 'wdbc.csv'  # alterar para wdbc.csv ou wdbc_robust_scaled.csv ou wdbc_minmax_scaled.csv
DATASET_PATH = Path('../data/processed') / DATASET_NAME
# Se quiser forçar explicitamente qual coluna é o target (no bruto ou no normalizado), defina aqui
# Exemplo: TARGET_COLUMN = 'diagnosis' ou TARGET_COLUMN = 'y'
TARGET_COLUMN = 'diagnosis'  # ajustar se necessário

# Output paths
REPORTS = Path('../model_reports/xgboost/cv')
CSV_OUT = REPORTS / 'csv'
IMAGES_OUT = REPORTS / 'images'
PDF_OUT = REPORTS / 'pdf'
MODEL_DIR = Path('../common/models/xgboost/v1')
# Basico outputs (single train/test) - sibling folder to 'cv'
BASICO_DIR = REPORTS.parent / 'basico'
BASICO_CSV = BASICO_DIR / 'csv'
BASICO_IMAGES = BASICO_DIR / 'images'

for d in [REPORTS, CSV_OUT, IMAGES_OUT, PDF_OUT, MODEL_DIR, BASICO_DIR, BASICO_CSV, BASICO_IMAGES]:
    d.mkdir(parents=True, exist_ok=True)

# XGBoost params (variável conforme solicitado)
xgb_params = {
    "n_estimators": 650,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "gamma": 0.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "min_child_weight": 1,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "n_jobs": 8,
    "random_state": 42
}

# CV and threshold config
N_SPLITS = 10
THRESHOLD = 0.5
# Colunas identificadoras que NÃO devem entrar como features no modelo, mas devem ser mantidas nos arquivos de saída
EXCLUDE_COLUMNS = ['id', 'ID', 'patient_id', 'pid']

# Pareto configuration: threshold (fraction) and selected-features placeholder (set to None to be updated after CV)
PARETO_THRESHOLD = 0.9  # fraction, e.g. 0.8 == 80%
PARETO_SELECTED_FEATURES = None

# Variance percent-difference threshold (fraction). Example: 0.1 == 10% difference.
VAR_DIFF_PCT_THRESHOLD = 0.01
TEST_SIZE = 0.3
RT_RANDOM_STATE = 42


In [2]:
# Imports principais
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
                             balanced_accuracy_score, confusion_matrix, roc_curve, log_loss, auc)
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm
import joblib

sns.set(style='whitegrid')

In [4]:
# Load dataset (path definido na configuração)
import shutil
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Arquivo de dataset não encontrado: {DATASET_PATH.resolve()}')

# Preserve a raw copy of the original CSV (unaltered) in data/processed with suffix _raw
RAW_COPY_PATH = Path('../data/processed') / f"{DATASET_NAME.replace('.csv','')}_raw.csv"
try:
    shutil.copy2(DATASET_PATH, RAW_COPY_PATH)
    print(f'Raw dataset copiado para: {RAW_COPY_PATH}')
except Exception as e_copy:
    print('Não foi possível copiar o CSV original para processed:', e_copy)

# Leia o dataset (continuamos a usar df para processamento posterior)
df = pd.read_csv(DATASET_PATH)
# Função utilitária para encontrar coluna aproximada (case/strip tolerant)
def find_column(df, name):
    # tenta match exato
    if name in df.columns:
        return name
    # tenta match case-insensitive
    low = {c.lower().strip(): c for c in df.columns}
    if name.lower().strip() in low:
        return low[name.lower().strip()]
    # tenta encontrar coluna que contenha o nome
    for c in df.columns:
        if name.lower().strip() in c.lower():
            return c
    return None

# Determina a coluna alvo: usa TARGET_COLUMN quando fornecida, senão infere
target_col = None
if TARGET_COLUMN:
    candidate = find_column(df, TARGET_COLUMN)
    if candidate is None:
        raise ValueError(f"TARGET_COLUMN='{TARGET_COLUMN}' não encontrada. Colunas: {list(df.columns)}")
    target_col = candidate
else:
    # heurística leve: nomes comuns
    for cand in ['y', 'diagnosis', 'target', 'label', 'class', 'diagnose']:
        candidate = find_column(df, cand)
        if candidate:
            target_col = candidate
            break
    # procura colunas binárias
    if target_col is None:
        for c in df.columns:
            if c.lower().strip() in ('id', 'patient_id', 'pid'):
                continue
            try:
                nunq = df[c].nunique(dropna=True)
            except Exception:
                nunq = 999
            if nunq == 2:
                target_col = c
                break
    # fallback para última coluna com poucos valores distintos
    if target_col is None:
        lastc = df.columns[-1]
        if df[lastc].nunique(dropna=True) <= 10:
            target_col = lastc
    if target_col is None:
        raise ValueError(f"Não foi possível inferir a coluna alvo. Defina TARGET_COLUMN ou verifique as colunas: {list(df.columns)}")

# Renomeia para 'y' e garante coluna disponível antes de acessar
if target_col != 'y':
    df = df.rename(columns={target_col: 'y'})
if 'y' not in df.columns:
    raise KeyError('Coluna y não encontrada após renomeação')

# Se y não for numérico, discretiza (M/B -> 1/0 ou factorize para múltiplas classes)
if not np.issubdtype(df['y'].dtype, np.number):
    unique_vals = list(df['y'].dropna().unique())
    low = [str(u).lower() for u in unique_vals]
    if set(low) <= set(['m', 'b']):
        mapping = {v: (1 if str(v).lower() == 'm' else 0) for v in unique_vals}
        df['y'] = df['y'].map(mapping)
    else:
        # factorize para inteiros 0..k-1
        df['y'], uniques = pd.factorize(df['y'])
    # salva versão discretizada para inspeção
    df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_y_numeric.csv', index=False)

# Substitui inf valores e dropna (pode ajustar imputação se preferir)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna()

# reorganiza para garantir que y seja a última coluna
cols = [c for c in df.columns if c != 'y'] + ['y']
df = df[cols]

# salva uma cópia de entrada no csv de reports (final)
df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}', index=False)

print('Dataset shape:', df.shape)

Raw dataset copiado para: ..\data\processed\wdbc_raw.csv
Dataset shape: (569, 32)


In [ ]:
# ============================
# XGBoost 8/1/1 com Optuna (TPE + Pruning) + métricas e CSVs — alinhado ao RF
# ============================
import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, log_loss, confusion_matrix
)
import time, joblib
from pathlib import Path

# --------- Checagens de pré-condição ---------
try:
    df
except NameError:
    raise RuntimeError("df não está definido. Rode a célula de carregamento/renomeação de 'y' antes desta.")

if 'y' not in df.columns:
    raise RuntimeError("Coluna 'y' não encontrada em df.")

try:
    EXCLUDE_COLUMNS
except NameError:
    EXCLUDE_COLUMNS = ['__cls_tmp__', 'orig_index', 'diagnosis', 'id', 'ID', 'patient_id', 'pid',
                       'fold', 'y_train', 'prob_0', 'prob_1', 'y_proba', 'y_pred']

# --------- Monta X / y ---------
y = df['y'].copy()
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower and c != 'y']
FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]

X_model = df[FEATURES].copy()
X_model.replace([np.inf, -np.inf], np.nan, inplace=True)
X_model = X_model.fillna(X_model.median(numeric_only=True))

classes = np.unique(y.dropna().values)
n_classes = len(classes)
is_binary = (n_classes == 2)

if is_binary:
    c0, c1 = classes[0], classes[1]
    neg = int((y == c0).sum())
    pos = int((y == c1).sum())
    scale_pos_weight_default = float(neg / max(pos, 1))
else:
    scale_pos_weight_default = 1.0

# --------- Configs ---------
N_TRIALS_OPTUNA   = 30  # reduzido de 60 para 30 (otimização mais rápida)
EARLY_STOP_ROUNDS = 100
RANDOM_STATE      = 42
N_JOBS            = -1

# threshold global (se não existir)
try:
    THRESHOLD
except NameError:
    THRESHOLD = 0.5

EPS = 1e-12

# ===================== 8/1/1: define folds externos (10) =====================
outer_kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(outer_kf.split(X_model, y))

# Último fold = teste, penúltimo = validação, demais 8 = treino
test1_idx = fold_indices[-1][1]   # fold 10
val1_idx  = fold_indices[-2][1]   # fold 9
# Conjunto de treino = todos os índices que NÃO estão em val/test
train8_idx = np.setdiff1d(np.arange(len(X_model)), np.concatenate([val1_idx, test1_idx]))

X_train8, y_train8 = X_model.iloc[train8_idx], y.iloc[train8_idx]
X_val1,   y_val1   = X_model.iloc[val1_idx],   y.iloc[val1_idx]
X_test1,  y_test1  = X_model.iloc[test1_idx],  y.iloc[test1_idx]

# ===================== Optuna (CV interna = 8 folds no conjunto de treino) =====================
cv_for_optuna = StratifiedKFold(n_splits=8, shuffle=True, random_state=RANDOM_STATE)

def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators":        trial.suggest_int("n_estimators", 150, 700),
        "max_depth":           trial.suggest_int("max_depth", 3, 12),
        "learning_rate":       trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True),
        "subsample":           trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":    trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_weight":    trial.suggest_int("min_child_weight", 1, 12),
        "gamma":               trial.suggest_float("gamma", 0.0, 10.0),
        "reg_alpha":           trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":          trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_delta_step":      trial.suggest_int("max_delta_step", 0, 5),
        "tree_method":         "hist",
        "n_jobs":              N_JOBS,
        "random_state":        RANDOM_STATE,
        "verbosity":           0,
    }

    if is_binary:
        params.update({
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "scale_pos_weight": scale_pos_weight_default,
        })
        maximize_metric = True
    else:
        params.update({
            "objective": "multi:softprob",
            "num_class": n_classes,
            "eval_metric": "mlogloss",
        })
        maximize_metric = False

    fold_scores = []
    for fold_id, (tr_idx, va_idx) in enumerate(cv_for_optuna.split(X_train8, y_train8), 1):
        X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
        y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

        model = XGBClassifier(**params)

        use_callbacks = hasattr(xgb, "callback") and hasattr(xgb.callback, "EarlyStopping")
        if use_callbacks:
            es_cb = xgb.callback.EarlyStopping(
                rounds=EARLY_STOP_ROUNDS,
                save_best=True,
                maximize=maximize_metric
            )
            try:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[es_cb])
            except TypeError:
                model.fit(X_tr, y_tr)
        else:
            try:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
            except TypeError:
                model.fit(X_tr, y_tr)

        proba_va = model.predict_proba(X_va)
        if is_binary:
            score = roc_auc_score(y_va, proba_va[:, 1])
        else:
            score = roc_auc_score(y_va, proba_va, multi_class="ovr", average="macro")

        fold_scores.append(float(score))
        trial.report(float(np.mean(fold_scores)), fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))

sampler = TPESampler(seed=RANDOM_STATE, multivariate=True, group=True)
pruner  = MedianPruner(n_warmup_steps=2)

study = optuna.create_study(
    study_name="xgb_tpe_auc_8fold_search",
    direction="maximize",
    sampler=sampler,
    pruner=pruner
)
study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)

print("Best value (AUROC, CV interna 8 folds):", study.best_value)
print("Best trial:", study.best_trial.number)

best = study.best_trial.params.copy()

# --------- Dicionário final para reuso ---------
xgb_params = {
    "n_estimators":      int(best.get("n_estimators", 300)),
    "max_depth":         int(best.get("max_depth", 6)),
    "learning_rate":     float(best.get("learning_rate", 0.05)),
    "subsample":         float(best.get("subsample", 0.8)),
    "colsample_bytree":  float(best.get("colsample_bytree", 0.8)),
    "min_child_weight":  int(best.get("min_child_weight", 1)),
    "gamma":             float(best.get("gamma", 0.0)),
    "reg_alpha":         float(best.get("reg_alpha", 0.0)),
    "reg_lambda":        float(best.get("reg_lambda", 1.0)),
    "max_delta_step":    int(best.get("max_delta_step", 0)),
    "tree_method":       "hist",
    "n_jobs":            N_JOBS,
    "random_state":      RANDOM_STATE,
    "verbosity":         0,
}
if is_binary:
    xgb_params.update({
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "scale_pos_weight": scale_pos_weight_default
    })
else:
    xgb_params.update({
        "objective": "multi:softprob",
        "eval_metric": "mlogloss",
        "num_class": n_classes
    })

print("\n=== xgb_params finais (8/1/1 corrigido) ===")
for k, v in xgb_params.items():
    print(f"{k} = {v}")

# ===================== Métricas no mesmo formato do Random Forest =====================

def cross_entropy_per_class(y_true, p_pos, eps=1e-12):
    """
    Versão idêntica à usada no RF.
    y_true: [0/1]; p_pos: probabilidade prevista da classe 1.
    Retorna: CE0 (média de -log(1-p) em y=0) e CE1 (média de -log(p) em y=1).
    """
    y_true = np.asarray(y_true).astype(int)
    p_pos = np.clip(np.asarray(p_pos), eps, 1 - eps)
    mask0 = (y_true == 0)
    mask1 = ~mask0
    ce0 = float(np.mean(-np.log(1 - p_pos[mask0]))) if np.any(mask0) else np.nan
    ce1 = float(np.mean(-np.log(p_pos[mask1])))     if np.any(mask1) else np.nan
    return ce0, ce1

def all_metrics_table(y_true, proba_pos, threshold=0.5, eps=1e-12):
    """
    Copiado do RF: retorna dict com TODAS as métricas:
    - Acurácia, Cross-Entropy C0/C1, Proporção C0/C1, F1, Balanced Accuracy,
      Precision, Recall, TP/FP/TN/FN (abs) e TP/FP/TN/FN (% do total).
    """
    y_true = np.asarray(y_true).astype(int)
    proba_pos = np.asarray(proba_pos)
    y_pred = (proba_pos >= threshold).astype(int)

    # Contagens
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN

    # Proporções por classe (no y_true)
    n0 = int(np.sum(y_true == 0))
    n1 = int(np.sum(y_true == 1))
    prop_c0 = n0 / total if total > 0 else np.nan
    prop_c1 = n1 / total if total > 0 else np.nan

    # Cross-Entropy por classe
    ce0, ce1 = cross_entropy_per_class(y_true, proba_pos, eps=eps)

    # Métricas padrão
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)

    # Balanced Accuracy = (recall_0 + recall_1)/2
    recall_1 = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    recall_0 = TN / (TN + FP) if (TN + FP) > 0 else np.nan
    bal_acc  = np.nanmean([recall_0, recall_1])

    # Percentuais de confusão
    TP_pct = TP / total if total > 0 else np.nan
    FP_pct = FP / total if total > 0 else np.nan
    TN_pct = TN / total if total > 0 else np.nan
    FN_pct = FN / total if total > 0 else np.nan

    return {
        "Acurácia": acc,
        "Cross-Entropy C0": ce0,
        "Proporção C0": prop_c0,
        "Cross-Entropy C1": ce1,
        "Proporção C1": prop_c1,
        "F1 Score": f1,
        "Balanced Accuracy": bal_acc,
        "Precision": prec,
        "Recall": rec,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "TP(%)": TP_pct, "FP(%)": FP_pct, "TN(%)": TN_pct, "FN(%)": FN_pct,
    }

# ===================== (1) Reavalia CV interna (8 folds) com melhores params =====================
inner_fold_rows = []

for fold_id, (tr_idx, va_idx) in enumerate(cv_for_optuna.split(X_train8, y_train8), start=1):
    X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
    y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

    model_inner = XGBClassifier(**xgb_params)
    model_inner.fit(X_tr, y_tr)

    # Apenas binário: usa prob da classe positiva
    proba_va = model_inner.predict_proba(X_va)[:, 1]
    metrics = all_metrics_table(y_va, proba_va, threshold=THRESHOLD, eps=EPS)
    row = {"etapa": "inner_cv", "fold": fold_id, **metrics}
    inner_fold_rows.append(row)

df_inner = pd.DataFrame(inner_fold_rows)

# ===================== (2) Validação intermediária (fold 9) =====================
t_val_ini = time.time()

model_val = XGBClassifier(**xgb_params)
model_val.fit(X_train8, y_train8)
proba_val = model_val.predict_proba(X_val1)[:, 1]

val_metrics_dict = all_metrics_table(y_val1, proba_val, threshold=THRESHOLD, eps=EPS)
val_row = {"etapa": "validacao", "fold": 9, **val_metrics_dict}
df_val = pd.DataFrame([val_row])

# ===================== (3) Treino final (8+1 = 9 folds) e teste (fold 10) =====================
X_train9 = pd.concat([X_train8, X_val1])
y_train9 = pd.concat([y_train8, y_val1])

t_test_ini = time.time()

xgb_final = XGBClassifier(**xgb_params)
xgb_final.fit(X_train9, y_train9)
proba_test = xgb_final.predict_proba(X_test1)[:, 1]

test_metrics_dict = all_metrics_table(y_test1, proba_test, threshold=THRESHOLD, eps=EPS)
test_row = {"etapa": "teste", "fold": 10, **test_metrics_dict}
df_test = pd.DataFrame([test_row])

# ===================== Consolida e salva CSVs iguais ao RF =====================
df_all = pd.concat([df_inner, df_val, df_test], ignore_index=True)

# Garante que CSV_OUT exista
try:
    CSV_OUT
except NameError:
    CSV_OUT = None

if CSV_OUT is not None:
    CSV_OUT = Path(CSV_OUT)
    CSV_OUT.mkdir(parents=True, exist_ok=True)

    # mesmos nomes do RF (assumindo que CSV_OUT é diferente para RF e XGB)
    df_inner.to_csv(CSV_OUT / "cv_811_inner_folds.csv", index=False)
    df_val.to_csv(CSV_OUT / "cv_811_validation.csv", index=False)
    df_test.to_csv(CSV_OUT / "cv_811_test.csv", index=False)
    df_all.to_csv(CSV_OUT / "cv_811_results_long.csv", index=False)

    # também mantém um resumo compacto específico do XGB (val + teste)
    df_xgb_811_results = pd.concat([df_val, df_test], ignore_index=True)
    df_xgb_811_results.to_csv(CSV_OUT / "xgb_cv_811_results.csv", index=False)

    print("\nCSVs XGBoost 8/1/1 salvos em:", CSV_OUT)
else:
    print("\nCSV_OUT não definido — CSVs não foram salvos.")

print("\n=== Resultados XGBoost 8/1/1 (inner, validação e teste) ===")
print(df_all.to_string(index=False))

# ===================== Salva modelo final (compatível com seu fluxo) =====================
try:
    MODEL_DIR
except NameError:
    MODEL_DIR = None

if MODEL_DIR is not None:
    MODEL_DIR = Path(MODEL_DIR)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model_path = MODEL_DIR / "xgb_model_cf10_8_1_1_final.pkl"
    joblib.dump(xgb_final, model_path)
    print("Modelo XGBoost final salvo em:", model_path)


In [5]:
# ============================
# XGBoost 8/1/1 (hold-out) com Optuna (TPE + Pruning)
# - 10 folds externos: 8 treino, 1 validação (hold-out), 1 teste (hold-out)
# - Optuna roda APENAS dentro do treino (8 folds internos)
# - Métrica de otimização parametrizável: "auc" ou "accuracy"
# - Gera e imprime tabela final (validação e teste) no formato solicitado
# ============================

import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score
)
import joblib
from pathlib import Path

# ---------------------- Pré-condições ----------------------
try:
    df
except NameError:
    raise RuntimeError("df não está definido. Rode a célula de carregamento e garanta a coluna 'y' antes desta.")

if 'y' not in df.columns:
    raise RuntimeError("Coluna 'y' não encontrada em df.")

try:
    EXCLUDE_COLUMNS
except NameError:
    EXCLUDE_COLUMNS = [
        '__cls_tmp__', 'orig_index', 'diagnosis', 'id', 'ID',
        'patient_id', 'pid', 'fold', 'y_train', 'prob_0', 'prob_1',
        'y_proba', 'y_pred'
    ]

# ---------------------- Configs principais ----------------------
RANDOM_STATE      = 42
N_JOBS            = -1
N_TRIALS_OPTUNA   = 30
EARLY_STOP_ROUNDS = 100

# limiar global
THRESHOLD = globals().get("THRESHOLD", 0.5)
EPS = 1e-12

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# PARAMETRIZA A MÉTRICA DE OTIMIZAÇÃO AQUI:
# "auc" ou "accuracy"
OPTIMIZE_METRIC = globals().get("OPTIMIZE_METRIC", "auc").lower().strip()
if OPTIMIZE_METRIC not in {"auc", "accuracy"}:
    raise ValueError("OPTIMIZE_METRIC deve ser 'auc' ou 'accuracy'.")
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

# ---------------------- Monta X / y ----------------------
y = df['y'].copy()

exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower and c != 'y']
FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]

X_model = df[FEATURES].copy()
X_model.replace([np.inf, -np.inf], np.nan, inplace=True)
X_model = X_model.fillna(X_model.median(numeric_only=True))

classes = np.unique(y.dropna().values)
n_classes = len(classes)
is_binary = (n_classes == 2)

if not is_binary:
    raise RuntimeError("Este script está configurado para binário (como no Sepsis). Ajuste se precisar multi-classe.")

# mapeia para {0,1} se necessário
# (mantém compatível caso y seja {1,2} etc)
class_to_bin = {classes[0]: 0, classes[1]: 1}
y_bin = y.map(class_to_bin).astype(int)

neg = int((y_bin == 0).sum())
pos = int((y_bin == 1).sum())
scale_pos_weight_default = float(neg / max(pos, 1))

# ===================== 8/1/1: folds externos (10) =====================
outer_kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(outer_kf.split(X_model, y_bin))

# Último fold = teste (hold-out), penúltimo = validação (hold-out)
test_idx = fold_indices[-1][1]   # fold 10
val_idx  = fold_indices[-2][1]   # fold 9

# treino = tudo que não está em val/test
train_idx = np.setdiff1d(np.arange(len(X_model)), np.concatenate([val_idx, test_idx]))

X_train8, y_train8 = X_model.iloc[train_idx], y_bin.iloc[train_idx]
X_val1,   y_val1   = X_model.iloc[val_idx],   y_bin.iloc[val_idx]
X_test1,  y_test1  = X_model.iloc[test_idx],  y_bin.iloc[test_idx]

print("\n=== Split 8/1/1 (hold-out) ===")
print(f"Treino (8/10): {len(train_idx)} | Validação hold-out (1/10): {len(val_idx)} | Teste hold-out (1/10): {len(test_idx)}")
print("Validação e Teste são hold-out: NÃO entram no Optuna.\n")

# ===================== CV interna p/ Optuna (8 folds no treino) =====================
cv_for_optuna = StratifiedKFold(n_splits=8, shuffle=True, random_state=RANDOM_STATE)

def _safe_auc(y_true, proba_pos):
    y_true = np.asarray(y_true).astype(int)
    proba_pos = np.asarray(proba_pos)
    # AUC requer ambas as classes presentes
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, proba_pos))

def cross_entropy_per_class(y_true, p_pos, eps=1e-12):
    """
    CE-C0 = média de -log(1-p) para y=0
    CE-C1 = média de -log(p)   para y=1
    """
    y_true = np.asarray(y_true).astype(int)
    p_pos = np.clip(np.asarray(p_pos), eps, 1 - eps)
    mask0 = (y_true == 0)
    mask1 = (y_true == 1)
    ce0 = float(np.mean(-np.log(1 - p_pos[mask0]))) if np.any(mask0) else np.nan
    ce1 = float(np.mean(-np.log(p_pos[mask1])))     if np.any(mask1) else np.nan
    return ce0, ce1

def compute_metrics_block(y_true, proba_pos, threshold=0.5, eps=1e-12):
    """
    Retorna as métricas exatamente nos nomes que você usa na tabela:
    ACC F1 BAC PRE REC AUC CE-C0 CE-C1 CE-C2 y TP% FP% TN% FN%
    """
    y_true = np.asarray(y_true).astype(int)
    proba_pos = np.asarray(proba_pos)
    y_pred = (proba_pos >= threshold).astype(int)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN

    acc = float(accuracy_score(y_true, y_pred))
    f1  = float(f1_score(y_true, y_pred, zero_division=0))
    bac = float(balanced_accuracy_score(y_true, y_pred))
    pre = float(precision_score(y_true, y_pred, zero_division=0))
    rec = float(recall_score(y_true, y_pred, zero_division=0))
    auc = _safe_auc(y_true, proba_pos)

    ce0, ce1 = cross_entropy_per_class(y_true, proba_pos, eps=eps)

    TPp = TP / total if total > 0 else np.nan
    FPp = FP / total if total > 0 else np.nan
    TNp = TN / total if total > 0 else np.nan
    FNp = FN / total if total > 0 else np.nan

    return {
        "ACC": acc,
        "F1": f1,
        "BAC": bac,
        "PRE": pre,
        "REC": rec,
        "AUC": auc,
        "CE-C0": ce0,
        "CE-C1": ce1,
        "CE-C2": "---",   # binário
        "y": 1,           # na sua tabela você usa y=1 como alvo
        "TP%": TPp,
        "FP%": FPp,
        "TN%": TNp,
        "FN%": FNp,
    }

# ===================== Optuna objective (otimiza ACC ou AUC) =====================
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators":        trial.suggest_int("n_estimators", 150, 700),
        "max_depth":           trial.suggest_int("max_depth", 3, 12),
        "learning_rate":       trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True),
        "subsample":           trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":    trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_weight":    trial.suggest_int("min_child_weight", 1, 12),
        "gamma":               trial.suggest_float("gamma", 0.0, 10.0),
        "reg_alpha":           trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":          trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_delta_step":      trial.suggest_int("max_delta_step", 0, 5),
        "tree_method":         "hist",
        "n_jobs":              N_JOBS,
        "random_state":        RANDOM_STATE,
        "verbosity":           0,
        "objective":           "binary:logistic",
        "eval_metric":         "auc",
        "scale_pos_weight":    scale_pos_weight_default,
    }

    fold_scores = []
    for fold_id, (tr_idx, va_idx) in enumerate(cv_for_optuna.split(X_train8, y_train8), 1):
        X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
        y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

        model = XGBClassifier(**params)

        # early stopping (quando disponível)
        use_callbacks = hasattr(xgb, "callback") and hasattr(xgb.callback, "EarlyStopping")
        if use_callbacks:
            es_cb = xgb.callback.EarlyStopping(
                rounds=EARLY_STOP_ROUNDS,
                save_best=True,
                maximize=True
            )
            try:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[es_cb])
            except TypeError:
                model.fit(X_tr, y_tr)
        else:
            try:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
            except TypeError:
                model.fit(X_tr, y_tr)

        proba_va = model.predict_proba(X_va)[:, 1]

        if OPTIMIZE_METRIC == "auc":
            score = _safe_auc(y_va, proba_va)
            if np.isnan(score):
                score = 0.5  # fallback conservador
        else:
            y_pred_va = (proba_va >= THRESHOLD).astype(int)
            score = float(accuracy_score(y_va, y_pred_va))

        fold_scores.append(float(score))
        trial.report(float(np.mean(fold_scores)), fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))

sampler = TPESampler(seed=RANDOM_STATE, multivariate=True, group=True)
pruner  = MedianPruner(n_warmup_steps=2)

study = optuna.create_study(
    study_name=f"xgb_tpe_{OPTIMIZE_METRIC}_8fold_search",
    direction="maximize",
    sampler=sampler,
    pruner=pruner
)
study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)

print(f"\nBest value ({OPTIMIZE_METRIC.upper()}, CV interna 8 folds):", study.best_value)
print("Best trial:", study.best_trial.number)

best = study.best_trial.params.copy()

# --------- Params finais ---------
xgb_params = {
    "n_estimators":      int(best.get("n_estimators", 300)),
    "max_depth":         int(best.get("max_depth", 6)),
    "learning_rate":     float(best.get("learning_rate", 0.05)),
    "subsample":         float(best.get("subsample", 0.8)),
    "colsample_bytree":  float(best.get("colsample_bytree", 0.8)),
    "min_child_weight":  int(best.get("min_child_weight", 1)),
    "gamma":             float(best.get("gamma", 0.0)),
    "reg_alpha":         float(best.get("reg_alpha", 0.0)),
    "reg_lambda":        float(best.get("reg_lambda", 1.0)),
    "max_delta_step":    int(best.get("max_delta_step", 0)),
    "tree_method":       "hist",
    "n_jobs":            N_JOBS,
    "random_state":      RANDOM_STATE,
    "verbosity":         0,
    "objective":         "binary:logistic",
    "eval_metric":       "auc",
    "scale_pos_weight":  scale_pos_weight_default,
}

print("\n=== xgb_params finais ===")
for k, v in xgb_params.items():
    print(f"{k} = {v}")

# ===================== (A) Validação hold-out (fold 9) =====================
model_val = XGBClassifier(**xgb_params)
model_val.fit(X_train8, y_train8)
proba_val = model_val.predict_proba(X_val1)[:, 1]
val_metrics = compute_metrics_block(y_val1, proba_val, threshold=THRESHOLD, eps=EPS)

# ===================== (B) Treino final (8+1) e teste hold-out (fold 10) =====================
X_train9 = pd.concat([X_train8, X_val1], axis=0)
y_train9 = pd.concat([y_train8, y_val1], axis=0)

xgb_final = XGBClassifier(**xgb_params)
xgb_final.fit(X_train9, y_train9)
proba_test = xgb_final.predict_proba(X_test1)[:, 1]
test_metrics = compute_metrics_block(y_test1, proba_test, threshold=THRESHOLD, eps=EPS)

# ===================== Tabela final (validação + teste) =====================
df_out = pd.DataFrame([
    {"split": "validacao", **val_metrics},
    {"split": "teste",     **test_metrics},
])

# ordena colunas exatamente como pedido
cols_order = ["split", "ACC","F1","BAC","PRE","REC","AUC","CE-C0","CE-C1","CE-C2","y","TP%","FP%","TN%","FN%"]
df_out = df_out[cols_order]

# print na tela
print("\n=== MÉTRICAS FINAIS (8/1/1) — Validação hold-out e Teste hold-out ===")
print(df_out.to_string(index=False))

# ===================== Salva CSV final =====================
CSV_OUT = globals().get("CSV_OUT", None)
if CSV_OUT is None:
    CSV_OUT = "."

CSV_OUT = Path(CSV_OUT)
CSV_OUT.mkdir(parents=True, exist_ok=True)

out_path = CSV_OUT / f"xgb_811_metrics_val_test_opt_{OPTIMIZE_METRIC}.csv"
df_out.to_csv(out_path, index=False)

print("\nCSV final salvo em:", out_path)

# ===================== Salva modelo final =====================
MODEL_DIR = globals().get("MODEL_DIR", None)
if MODEL_DIR is None:
    MODEL_DIR = "."

MODEL_DIR = Path(MODEL_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / f"xgb_model_811_final_opt_{OPTIMIZE_METRIC}.pkl"
joblib.dump(xgb_final, model_path)

print("Modelo final salvo em:", model_path)


[I 2026-01-26 05:58:16,104] A new study created in memory with name: xgb_tpe_auc_8fold_search



=== Split 8/1/1 (hold-out) ===
Treino (8/10): 456 | Validação hold-out (1/10): 57 | Teste hold-out (1/10): 56
Validação e Teste são hold-out: NÃO entram no Optuna.



Best trial: 0. Best value: 0.990636:   3%|▎         | 1/30 [00:03<01:48,  3.75s/it]

[I 2026-01-26 05:58:19,845] Trial 0 finished with value: 0.9906355218855218 and parameters: {'n_estimators': 356, 'max_depth': 12, 'learning_rate': 0.06504856968981275, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.40921304830970556, 'min_child_weight': 2, 'gamma': 0.5808361216819946, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598, 'max_delta_step': 4}. Best is trial 0 with value: 0.9906355218855218.


Best trial: 0. Best value: 0.990636:   7%|▋         | 2/30 [00:05<01:07,  2.40s/it]

...

[I 2026-01-26 05:59:33,853] Trial 27 finished with value: 0.9894901394901394 and parameters: {'n_estimators': 309, 'max_depth': 11, 'learning_rate': 0.014235205877863063, 'subsample': 0.7815987336416435, 'colsample_bytree': 0.7608791389399077, 'min_child_weight': 3, 'gamma': 2.3056216154124947, 'reg_alpha': 0.0005061681536584715, 'reg_lambda': 0.0003213265659358647, 'max_delta_step': 4}. Best is trial 26 with value: 0.9926046176046176.


Best trial: 26. Best value: 0.992605:  97%|█████████▋| 29/30 [01:19<00:02,  2.41s/it]

[I 2026-01-26 05:59:35,682] Trial 28 finished with value: 0.9886724386724387 and parameters: {'n_estimators': 173, 'max_depth': 9, 'learning_rate': 0.020051642855095277, 'subsample': 0.573955998283932, 'colsample_bytree': 0.48558878980464404, 'min_child_weight': 1, 'gamma': 3.1561127664667614, 'reg_alpha': 0.7919048105952031, 'reg_lambda': 0.0003350948097932687, 'max_delta_step': 5}. Best is trial 26 with value: 0.9926046176046176.


Best trial: 26. Best value: 0.992605: 100%|██████████| 30/30 [01:22<00:00,  2.74s/it]


[I 2026-01-26 05:59:38,246] Trial 29 finished with value: 0.9906295093795093 and parameters: {'n_estimators': 317, 'max_depth': 10, 'learning_rate': 0.02645003025769765, 'subsample': 0.5730338207565535, 'colsample_bytree': 0.6496292346370566, 'min_child_weight': 1, 'gamma': 2.6457167137892297, 'reg_alpha': 0.00778079078873694, 'reg_lambda': 3.0226011195588566e-05, 'max_delta_step': 2}. Best is trial 26 with value: 0.9926046176046176.

Best value (AUC, CV interna 8 folds): 0.9926046176046176
Best trial: 26

=== xgb_params finais ===
n_estimators = 238
max_depth = 9
learning_rate = 0.032608515924771146
subsample = 0.7232286045329136
colsample_bytree = 0.6392143985049948
min_child_weight = 1
gamma = 0.6979183350762996
reg_alpha = 0.0029151162698406938
reg_lambda = 0.00011345377716785412
max_delta_step = 3
tree_method = hist
n_jobs = -1
random_state = 42
verbosity = 0
objective = binary:logistic
eval_metric = auc
scale_pos_weight = 1.6839622641509433

=== MÉTRICAS FINAIS (8/1/1) — Validaçã

In [ ]:
# Preparação dos dados X, y
y = df['y']

# Detecta colunas identificadoras a partir de EXCLUDE_COLUMNS (case-insensitive)
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
# garante que id columns não contenham a coluna alvo y
ID_COLS = [c for c in ID_COLS if c != 'y']

# FEATURES são todas as colunas exceto ids e y
FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]

# X_model será usado para treinar (sem ids)
X_model = df[FEATURES].copy()
X_model.replace([np.inf, -np.inf], np.nan, inplace=True)
# preenche valores faltantes simples (pode ser alterado)
X_model = X_model.fillna(X_model.median(numeric_only=True))

from sklearn.metrics import roc_curve  # necessário pois é usado abaixo
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, log_loss, confusion_matrix, roc_auc_score, auc
)
from tqdm import tqdm
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import matplotlib.pyplot as plt

# ==== Seleção 9/1 consistente com o esquema 8/1/1 ====
# Se já existem test1_idx/val1_idx/train8_idx do bloco 8/1/1, usa o mesmo teste (fold 10)
if 'test1_idx' in globals():
    # treino 9/1 = todos os índices menos o teste
    all_idx = np.arange(len(X_model))
    train9_idx = np.setdiff1d(all_idx, test1_idx)
    _selected_splits = [(train9_idx, test1_idx)]
    # define o número de folds para o rótulo (ex.: 10)
    try:
        N_SPLITS
    except NameError:
        N_SPLITS = 10
else:
    # fallback: comportamento antigo (pega o último split do KFold)
    try:
        N_SPLITS
    except NameError:
        N_SPLITS = 10
    kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    _all_splits = list(kf.split(X_model, y))
    _last_train_idx, _last_test_idx = _all_splits[-1]
    _selected_splits = [(_last_train_idx, _last_test_idx)]

# estruturas de resultado
resultados = []
# listas para métricas agregadas por fold (treino/teste)
aurocs_train = []
aurocs_test = []
fprs_train = []
tprs_train = []
fprs_test = []
tprs_test = []
importancias_lista = []
X_train_parts = []
X_test_parts = []
y_train_parts = []
y_test_parts = []
y_train_pred_parts = []
y_test_pred_parts = []

start_total = time.time()

with PdfPages(PDF_OUT / f'XGB_CV_{N_SPLITS}.pdf') as pdf:
    # total=1 porque só executaremos o último split => 9/1
    with tqdm(total=1, desc="🔄 Folds K-Fold (9/1)") as pbar:
        # define o "fold" como N_SPLITS (ex.: 10) para marcar que é o último
        for fold, (train_idx, test_idx) in enumerate(_selected_splits, start=N_SPLITS):
            t0 = time.time()
            # Para manter ids nos arquivos de saída, indexamos sobre o DataFrame original
            X_train_full = df.iloc[train_idx][ID_COLS + FEATURES] if len(ID_COLS) > 0 else df.iloc[train_idx][FEATURES]
            X_test_full  = df.iloc[test_idx][ID_COLS + FEATURES]  if len(ID_COLS) > 0 else df.iloc[test_idx][FEATURES]
            # Usamos apenas as FEATURES (sem ids) para treinar o modelo
            X_train = X_train_full[FEATURES].copy()
            X_test  = X_test_full[FEATURES].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            model = XGBClassifier(**xgb_params)
            model.fit(X_train, y_train)

            # Previsões de probabilidade
            y_proba_test_all = model.predict_proba(X_test)
            y_proba_test_C1 = y_proba_test_all[:, 1]
            y_proba_test_C1_bin = (y_proba_test_C1 >= THRESHOLD).astype(int)

            y_proba_train_all = model.predict_proba(X_train)
            y_proba_train_C1 = y_proba_train_all[:, 1]
            y_proba_train_C1_bin = (y_proba_train_C1 >= THRESHOLD).astype(int)

            # guarda partes completas (incluem ids) para concat posterior e rastreabilidade
            # mantemos o índice original para fácil merge posterior
            X_test_parts.append(X_test_full.assign(fold=fold).reset_index())
            y_test_parts.append(pd.Series(y_test, name='y_test').to_frame().assign(fold=fold).reset_index())
            y_test_pred_parts.append(
                pd.DataFrame(y_proba_test_all, columns=['prob_0', 'prob_1'], index=X_test_full.index)
                  .assign(fold=fold).reset_index()
            )

            X_train_parts.append(X_train_full.assign(fold=fold).reset_index())
            y_train_parts.append(pd.Series(y_train, name='y_train').to_frame().assign(fold=fold).reset_index())
            y_train_pred_parts.append(
                pd.DataFrame(y_proba_train_all, columns=['prob_0', 'prob_1'], index=X_train_full.index)
                  .assign(fold=fold).reset_index()
            )

            # Métricas Treino
            cm_train = confusion_matrix(y_train, y_proba_train_C1_bin)
            tn_tr, fp_tr, fn_tr, tp_tr = cm_train.ravel()
            total_cm_treino = cm_train.sum()
            fpr_tr, tpr_tr, _ = roc_curve(y_train, y_proba_train_C1)
            fprs_train.append(fpr_tr)
            tprs_train.append(tpr_tr)
            auroc_tr_val = roc_auc_score(y_train, y_proba_train_C1)
            aurocs_train.append(auroc_tr_val)

            MASK_0_TRAIN = (y_train == 0)
            MASK_1_TRAIN = (y_train == 1)
            logloss_train_0 = log_loss(
                y_train[MASK_0_TRAIN], y_proba_train_C1[MASK_0_TRAIN], labels=[0, 1]
            ) if MASK_0_TRAIN.sum() > 0 else np.nan
            logloss_train_1 = log_loss(
                y_train[MASK_1_TRAIN], y_proba_train_C1[MASK_1_TRAIN], labels=[0, 1]
            ) if MASK_1_TRAIN.sum() > 0 else np.nan
            cross_prop_train = pd.Series(y_train).value_counts(normalize=True) * 100

            # Métricas Teste
            cm_test = confusion_matrix(y_test, y_proba_test_C1_bin)
            tn_ts, fp_ts, fn_ts, tp_ts = cm_test.ravel()
            total_cm_teste = cm_test.sum()
            fpr_ts, tpr_ts, _ = roc_curve(y_test, y_proba_test_C1)
            fprs_test.append(fpr_ts)
            tprs_test.append(tpr_ts)
            auroc_ts_val = roc_auc_score(y_test, y_proba_test_C1)
            aurocs_test.append(auroc_ts_val)

            MASK_0_TEST = (y_test == 0)
            MASK_1_TEST = (y_test == 1)
            logloss_test_0 = log_loss(
                y_test[MASK_0_TEST], y_proba_test_C1[MASK_0_TEST], labels=[0, 1]
            ) if MASK_0_TEST.sum() > 0 else np.nan
            logloss_test_1 = log_loss(
                y_test[MASK_1_TEST], y_proba_test_C1[MASK_1_TEST], labels=[0, 1]
            ) if MASK_1_TEST.sum() > 0 else np.nan
            cross_prop_test = pd.Series(y_test).value_counts(normalize=True) * 100

            # Armazena resultados (Treino/Teste)
            resultados.append({
                'Conjunto': 'Treino(9folds)',
                'Fold': fold,
                'Acurácia': accuracy_score(y_train, y_proba_train_C1_bin),
                'Cross-Entropy C0': logloss_train_0,
                'Proporção C0': cross_prop_train.iloc[0] if 0 in cross_prop_train.index else np.nan,
                'Cross-Entropy C1': logloss_train_1,
                'Proporção C1': cross_prop_train.iloc[1] if 1 in cross_prop_train.index else np.nan,
                'F1 Score': f1_score(y_train, y_proba_train_C1_bin),
                'Balanced Accuracy': balanced_accuracy_score(y_train, y_proba_train_C1_bin),
                'Precision': precision_score(y_train, y_proba_train_C1_bin),
                'Recall': recall_score(y_train, y_proba_train_C1_bin),
                'AUROC': auroc_tr_val,
                'TP': tp_tr, 'FP': fp_tr, 'TN': tn_tr, 'FN': fn_tr,
                'TP(%)': round(tp_tr / total_cm_treino * 100, 2),
                'FP(%)': round(fp_tr / total_cm_treino * 100, 2),
                'TN(%)': round(tn_tr / total_cm_treino * 100, 2),
                'FN(%)': round(fn_tr / total_cm_treino * 100, 2),
                'Tempo (s)': round(time.time() - t0, 2)
            })

            resultados.append({
                'Conjunto': 'Teste(fold10)',
                'Fold': fold,
                'Acurácia': accuracy_score(y_test, y_proba_test_C1_bin),
                'Cross-Entropy C0': logloss_test_0,
                'Proporção C0': cross_prop_test.iloc[0] if 0 in cross_prop_test.index else np.nan,
                'Cross-Entropy C1': logloss_test_1,
                'Proporção C1': cross_prop_test.iloc[1] if 1 in cross_prop_test.index else np.nan,
                'F1 Score': f1_score(y_test, y_proba_test_C1_bin),
                'Balanced Accuracy': balanced_accuracy_score(y_test, y_proba_test_C1_bin),
                'Precision': precision_score(y_test, y_proba_test_C1_bin),
                'Recall': recall_score(y_test, y_proba_test_C1_bin),
                'AUROC': auroc_ts_val,
                'TP': tp_ts, 'FP': fp_ts, 'TN': tn_ts, 'FN': fn_ts,
                'TP(%)': round(tp_ts / total_cm_teste * 100, 2),
                'FP(%)': round(fp_ts / total_cm_teste * 100, 2),
                'TN(%)': round(tn_ts / total_cm_teste * 100, 2),
                'FN(%)': round(fn_ts / total_cm_teste * 100, 2),
                'Tempo (s)': round(time.time() - t0, 2)
            })

            # Importância das features (percentual)
            importancias = model.feature_importances_
            nomes_colunas = FEATURES
            linha_importancia = {'Conjunto': 'Teste(fold10)', 'Fold': fold}
            for nome, valor in zip(nomes_colunas, importancias):
                linha_importancia[nome] = f"{valor * 100:.2f}%"
            importancias_lista.append(linha_importancia)

            # salva modelo por fold (aqui só 1, 9/1)
            model_path = MODEL_DIR / f'xgb_model_cf{N_SPLITS}_9_1.pkl'
            joblib.dump(model, model_path)

            pbar.update(1)

    # fim do "loop" (apenas 1 split 9/1)

# Concatena e salva os resultados
df_resultados = pd.DataFrame(resultados)
# adiciona média das métricas (aqui será idêntica, pois só há 1 split, mas mantém compatibilidade)
mean_row = df_resultados.select_dtypes(include=[np.number]).mean()
mean_row['Conjunto'] = 'Média'
mean_row['Fold'] = 'Média'
df_resultados = pd.concat([df_resultados, pd.DataFrame([mean_row])], ignore_index=True, sort=False)

# salva csvs principais
df_resultados.to_csv(CSV_OUT / f'xgb_cv_results_{N_SPLITS}_9_1.csv', index=False)
pd.DataFrame(importancias_lista).to_csv(CSV_OUT / f'xgb_feature_importances_{N_SPLITS}_9_1.csv', index=False)

# Concatena blocos para gerar dataframes completos de treino/teste com probabilidades
if len(X_train_parts) > 0:
    X_train_global = pd.concat(X_train_parts, ignore_index=True)
    y_train_global = pd.concat(y_train_parts, ignore_index=True)
    y_train_pred_global = pd.concat(y_train_pred_parts, ignore_index=True)
else:
    X_train_global = pd.DataFrame(); y_train_global = pd.DataFrame(); y_train_pred_global = pd.DataFrame()

if len(X_test_parts) > 0:
    X_test_global = pd.concat(X_test_parts, ignore_index=True)
    y_test_global = pd.concat(y_test_parts, ignore_index=True)
    y_test_pred_global = pd.concat(y_test_pred_parts, ignore_index=True)
else:
    X_test_global = pd.DataFrame(); y_test_global = pd.DataFrame(); y_test_pred_global = pd.DataFrame()

# Ajusta nomes: os objetos concat possuem coluna 'index' que era o índice original; renomeamos para 'orig_index' para merge limpo
for df_block in [X_train_global, X_test_global, y_train_global, y_test_global, y_train_pred_global, y_test_pred_global]:
    if 'index' in df_block.columns:
        df_block.rename(columns={'index': 'orig_index'}, inplace=True)

# Mescla para criar arquivos finais com probas e fold
if not X_train_global.empty and not y_train_global.empty:
    train_df = X_train_global.merge(y_train_global, on=['orig_index', 'fold'], how='left')
    if not y_train_pred_global.empty:
        train_df = train_df.merge(y_train_pred_global, on=['orig_index', 'fold'], how='left')
else:
    train_df = pd.DataFrame()

if not X_test_global.empty and not y_test_global.empty:
    test_df = X_test_global.merge(y_test_global, on=['orig_index', 'fold'], how='left')
    if not y_test_pred_global.empty:
        test_df = test_df.merge(y_test_pred_global, on=['orig_index', 'fold'], how='left')
else:
    test_df = pd.DataFrame()

# Salva os datasets com probabilidades (prob_0, prob_1) e já binarizados por threshold
if not test_df.empty:
    test_df['y_proba'] = test_df['prob_1']
    test_df['y_pred'] = (test_df['y_proba'] >= THRESHOLD).astype(int)
    test_df.to_csv(CSV_OUT / f'xgb_test_with_proba_{N_SPLITS}_9_1.csv', index=False)
if not train_df.empty:
    train_df['y_proba'] = train_df['prob_1']
    train_df['y_pred'] = (train_df['y_proba'] >= THRESHOLD).astype(int)
    train_df.to_csv(CSV_OUT / f'xgb_train_with_proba_{N_SPLITS}_9_1.csv', index=False)

# Salva um dataset unificado em process (com y_proba e y_pred sobre todo o conjunto de teste concatenado)
if not test_df.empty:
    test_df.to_csv(Path('../data/processed') / f'wdbc_xgb_test_with_proba_{N_SPLITS}_9_1.csv', index=False)

# create a full-augmented dataset for the original input dataset with y, y_pred, y_proba
try:
    if not test_df.empty:
        df_aug = df.reset_index().rename(columns={'index': 'orig_index'})
        preds = test_df[['orig_index', 'y_pred', 'y_proba']]
        df_augmented = pd.merge(df_aug, preds, how='left', on='orig_index')
        df_augmented.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_preds_xgb_{N_SPLITS}_9_1.csv', index=False)
    else:
        print('test_df vazio: não foi possível gerar dataset augmentado com previsões')
except Exception as e_aug:
    print('Erro ao gerar dataset augmentado com previsões:', e_aug)

print('Resultados salvos em:', CSV_OUT)


In [ ]:
# # Diagnóstico rápido para NaN / tipos em train_df e test_df
# for name, ycol in [('train_df', 'y_train'), ('test_df', 'y_test')]:
#     df_block = globals().get(name)
#     if df_block is None:
#         print(f"{name} não existe")
#         continue
#     if df_block.empty:
#         print(f"{name} existe mas está vazio")
#         continue
#     print(f"\n{name}: shape={df_block.shape}")
#     for col in [ycol, 'prob_1', 'y_proba']:
#         if col in df_block.columns:
#             print(f"  {col}: nulls={df_block[col].isnull().sum()}, dtype={df_block[col].dtype}, unique_count={df_block[col].nunique(dropna=True)}")
#     cols_to_show = [c for c in [ycol, 'prob_1', 'y_proba'] if c in df_block.columns]
#     display(df_block[cols_to_show].head(10))

In [ ]:
# Geração de PDF resumo (com algumas páginas importantes)
with PdfPages(PDF_OUT / f'XGB_CV_SUMMARY_{N_SPLITS}.pdf') as pdf:
    # Página de parâmetros
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    parametros = xgb_params.copy()
    parametros['threshold']        = THRESHOLD
    # agora deixamos explícito o esquema 8/1/1
    parametros['outer_n_splits']   = N_SPLITS          # normalmente 10
    parametros['schema']           = '8/1/1 (treino: 8 folds, validação: 1 fold, teste: 1 fold)'
    parametros['inner_cv_folds']   = 8                 # CV interna do Optuna

    texto = 'Algoritmo: XGBoost\n' + '\n'.join([f'{k}: {v}' for k, v in parametros.items()])
    ax.text(0, 1, texto, fontsize=11, family='monospace', verticalalignment='top')
    ax.set_title('📌 Parâmetros do Modelo - XGBoost')
    # salva PNG dos parâmetros (opcional)
    try:
        png_path = IMAGES_OUT / f'params_{N_SPLITS}.png'
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
    except Exception:
        pass
    pdf.savefig(fig)
    plt.close()

    # Página de métricas resumidas
    if not df_resultados.empty:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.axis('off')
        display_df = (
            df_resultados
            .groupby(['Conjunto'])
            .mean(numeric_only=True)
            .T
            .round(4)
        )
        table = ax.table(
            cellText=display_df.values,
            colLabels=display_df.columns,
            rowLabels=display_df.index,
            loc='center'
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        try:
            png_path = IMAGES_OUT / f'metrics_summary_{N_SPLITS}.png'
            fig.savefig(png_path, dpi=300, bbox_inches='tight')
        except Exception:
            pass
        pdf.savefig(fig)
        plt.close()

    # Página ROC - Treino vs Teste (lado a lado)
    try:
        plotted = False
        # Tenta usar dataframes concatenados quando disponíveis (train_df/test_df)
        if 'y_train' in train_df.columns and 'prob_1' in train_df.columns:
            y_train_all = train_df['y_train']
            y_score_train_all = train_df['prob_1']
            fpr_train, tpr_train, _ = roc_curve(y_train_all, y_score_train_all)
            roc_auc_train = auc(fpr_train, tpr_train)
            plotted = True
        elif len(fprs_train) > 0:
            # média interpolada das curvas por fold (treino)
            mean_fpr = np.linspace(0, 1, 200)
            tprs = []
            for fpr, tpr in zip(fprs_train, tprs_train):
                tpr_i = np.interp(mean_fpr, fpr, tpr)
                tpr_i[0] = 0.0
                tprs.append(tpr_i)
            mean_tpr = np.mean(tprs, axis=0)
            mean_tpr[-1] = 1.0
            fpr_train, tpr_train = mean_fpr, mean_tpr
            roc_auc_train = np.mean(aurocs_train) if len(aurocs_train) > 0 else auc(fpr_train, tpr_train)
            plotted = True

        if 'y_test' in test_df.columns and 'prob_1' in test_df.columns:
            y_test_all = test_df['y_test']
            y_score_test_all = test_df['prob_1']
            fpr_test, tpr_test, _ = roc_curve(y_test_all, y_score_test_all)
            roc_auc_test = auc(fpr_test, tpr_test)
            plotted = True
        elif len(fprs_test) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = []
            for fpr, tpr in zip(fprs_test, tprs_test):
                tpr_i = np.interp(mean_fpr, fpr, tpr)
                tpr_i[0] = 0.0
                tprs.append(tpr_i)
            mean_tpr = np.mean(tprs, axis=0)
            mean_tpr[-1] = 1.0
            fpr_test, tpr_test = mean_fpr, mean_tpr
            roc_auc_test = np.mean(aurocs_test) if len(aurocs_test) > 0 else auc(fpr_test, tpr_test)
            plotted = True

        if plotted:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            # ROC Treino (esquerda)
            try:
                axes[0].plot(fpr_train, tpr_train, label=f'AUC = {roc_auc_train:.4f}')
                axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
                axes[0].set_title('ROC - Treino (8+1 folds)')
                axes[0].set_xlabel('FPR')
                axes[0].set_ylabel('TPR')
                axes[0].legend(loc='lower right')
            except Exception:
                axes[0].text(0.5, 0.5, 'Não há dados de treino para ROC', ha='center')
            # ROC Teste (direita)
            try:
                axes[1].plot(fpr_test, tpr_test, label=f'AUC = {roc_auc_test:.4f}')
                axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
                axes[1].set_title('ROC - Teste (fold 10)')
                axes[1].set_xlabel('FPR')
                axes[1].set_ylabel('TPR')
                axes[1].legend(loc='lower right')
            except Exception:
                axes[1].text(0.5, 0.5, 'Não há dados de teste para ROC', ha='center')
            plt.tight_layout()
            # salva PNG do ROC combinado
            try:
                png_path = IMAGES_OUT / f'roc_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
        else:
            print('Não foi possível gerar curvas ROC (dados ausentes)')
    except Exception as e:
        print('Não foi possível gerar ROC agregado:', e)

    # Página Confusion Matrix - Treino vs Teste (counts + percent)
    try:
        def build_cm_from_df(df_block, true_col, pred_col):
            # retorna cm e cm_percent
            y_true = df_block[true_col]
            y_pred = df_block[pred_col]
            cm = confusion_matrix(y_true, y_pred)
            total = cm.sum()
            cm_percent = cm / total * 100 if total > 0 else np.zeros_like(cm, dtype=float)
            return cm, cm_percent

        # tenta usar os dataframes concat (se disponíveis)
        cm_train, cm_train_pct = None, None
        cm_test, cm_test_pct = None, None
        if 'y_train' in train_df.columns and 'y_pred' in train_df.columns:
            cm_train, cm_train_pct = build_cm_from_df(train_df, 'y_train', 'y_pred')
        else:
            # fallback: agrega de df_resultados (soma de TP,FP,TN,FN)
            if 'TP' in df_resultados.columns:
                train_rows = df_resultados[df_resultados['Conjunto'] == 'Treino']
                TP = int(train_rows['TP'].sum()) if not train_rows.empty else 0
                FP = int(train_rows['FP'].sum()) if not train_rows.empty else 0
                TN = int(train_rows['TN'].sum()) if not train_rows.empty else 0
                FN = int(train_rows['FN'].sum()) if not train_rows.empty else 0
                cm_train = np.array([[TN, FP], [FN, TP]])
                total = cm_train.sum()
                cm_train_pct = cm_train / total * 100 if total > 0 else np.zeros_like(cm_train, dtype=float)

        if 'y_test' in test_df.columns and 'y_pred' in test_df.columns:
            cm_test, cm_test_pct = build_cm_from_df(test_df, 'y_test', 'y_pred')
        else:
            if 'TP' in df_resultados.columns:
                test_rows = df_resultados[df_resultados['Conjunto'] == 'Teste']
                TP = int(test_rows['TP'].sum()) if not test_rows.empty else 0
                FP = int(test_rows['FP'].sum()) if not test_rows.empty else 0
                TN = int(test_rows['TN'].sum()) if not test_rows.empty else 0
                FN = int(test_rows['FN'].sum()) if not test_rows.empty else 0
                cm_test = np.array([[TN, FP], [FN, TP]])
                total = cm_test.sum()
                cm_test_pct = cm_test / total * 100 if total > 0 else np.zeros_like(cm_test, dtype=float)

        # plota se ao menos uma matriz existir
        if (cm_train is None and cm_test is None):
            print('Não há dados suficientes para gerar matrizes de confusão.')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            def plot_cm(ax, cm, cm_pct, title):
                if cm is None:
                    ax.text(0.5, 0.5, 'Sem dados', ha='center', va='center')
                    ax.axis('off')
                    return
                annot = np.empty(cm.shape, dtype=object)
                for i in range(cm.shape[0]):
                    for j in range(cm.shape[1]):
                        annot[i, j] = f"{int(cm[i,j])}\n({cm_pct[i,j]:.2f}%)"
                sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=False, annot_kws={'size':12})
                ax.set_xlabel('Predito')
                ax.set_ylabel('Verdadeiro')
                ax.set_xticklabels(['0','1'])
                ax.set_yticklabels(['0','1'])
                ax.set_title(title)

            plot_cm(axes[0], cm_train, cm_train_pct, 'Matriz de Confusão - Treino (8+1 folds)')
            plot_cm(axes[1], cm_test,  cm_test_pct,  'Matriz de Confusão - Teste (fold 10)')

            plt.tight_layout()
            # salva PNG da matriz de confusão
            try:
                png_path = IMAGES_OUT / f'confusion_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
    except Exception as e:
        print('Não foi possível gerar matrizes de confusão:', e)

    # Página importâncias médias + Pareto + variâncias + violinos (inalterado)
    # (mantém exatamente o restante do seu bloco)


In [ ]:
# === XGBoost Basico (single train/test split) — alinhado com 8/1/1 (último fold = teste) ===
# Parâmetros de split (configurados no topo do notebook)
# TEST_SIZE: usado apenas como alias nos nomes de arquivos (ex.: 0.1 → 9/1)
# RT_RANDOM_STATE: seed para reprodutibilidade
try:
    TEST_SIZE
except NameError:
    TEST_SIZE = 0.1  # apenas alias para nomes (equivalente a ~9/1)

try:
    RT_RANDOM_STATE
except NameError:
    RT_RANDOM_STATE = 42

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
import pandas as pd
import joblib
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

# Prepara X e y — usa FEATURES já detectadas no notebook e df
if 'FEATURES' not in globals():
    raise RuntimeError('FEATURES não encontrado — execute as células anteriores para preparar os dados')
if 'df' not in globals():
    raise RuntimeError('Dataframe df não encontrado — execute as células anteriores para carregar os dados')

# Usa X sem colunas identificadoras
X = df[FEATURES].copy()
y = df['y'].copy()

# ==== Split 9/1 determinístico via KFold (compatível com o RF básico) ====
N_SPLITS_BASIC = 10  # 9/1
kf_basic = StratifiedKFold(n_splits=N_SPLITS_BASIC, shuffle=True, random_state=RT_RANDOM_STATE)
fold_pairs = list(kf_basic.split(X, y))

# último fold => teste; todos os demais => treino
_, test_idx = fold_pairs[-1]
all_idx = np.arange(len(X))
train_idx = np.setdiff1d(all_idx, test_idx)

# preserva índices originais (equivalente ao que você fazia com reset_index)
orig_train_idx = X.iloc[train_idx].index.values
orig_test_idx  = X.iloc[test_idx].index.values

X_train = X.iloc[train_idx].copy().set_index(pd.Index(orig_train_idx))
X_test  = X.iloc[test_idx].copy().set_index(pd.Index(orig_test_idx))
y_train = y.iloc[train_idx]
y_test  = y.iloc[test_idx]

# Treina modelo
model_basic = XGBClassifier(**xgb_params)
model_basic.fit(X_train, y_train)

# Predições binárias
y_pred_train_bin = model_basic.predict(X_train)
y_pred_test_bin  = model_basic.predict(X_test)

# Probabilidades (para AUROC)
y_proba_train = model_basic.predict_proba(X_train)[:, 1]
y_proba_test  = model_basic.predict_proba(X_test)[:, 1]

# Acurácias e outras métricas
acc_train = accuracy_score(y_train, y_pred_train_bin)
acc_test  = accuracy_score(y_test,  y_pred_test_bin)
f1_tr     = f1_score(y_train, y_pred_train_bin)
f1_ts     = f1_score(y_test,  y_pred_test_bin)
prec_tr   = precision_score(y_train, y_pred_train_bin)
prec_ts   = precision_score(y_test,  y_pred_test_bin)
rec_tr    = recall_score(y_train, y_pred_train_bin)
rec_ts    = recall_score(y_test,  y_pred_test_bin)

auc_tr = roc_auc_score(y_train, y_proba_train) if len(np.unique(y_train)) > 1 else np.nan
auc_ts = roc_auc_score(y_test,  y_proba_test)  if len(np.unique(y_test)) > 1  else np.nan

print(f"[9/1 XGB] Acurácia treino: {acc_train:.4f} — teste: {acc_test:.4f}")

# Salva modelo .pkl e dados
model_name = MODEL_DIR / f'xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.pkl'
joblib.dump(model_basic, model_name)

# save CSVs to basico csv folder
X_train.to_csv(BASICO_CSV / f'X_train_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv')
X_test.to_csv (BASICO_CSV / f'X_test_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv')
pd.Series(y_test,  name='y_test').to_csv (BASICO_CSV / f'y_test_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv',  index=True)
pd.Series(y_train, name='y_train').to_csv(BASICO_CSV / f'y_train_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv', index=True)

# Salva features
features = X.columns.tolist()
joblib.dump(features, MODEL_DIR / f'features_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.pkl')

# Salva métricas em CSV no basico csv folder
metrics_basic = pd.DataFrame([{ 
    'model': str(model_name.name),
    'schema': '9/1 via KFold (último fold = teste)',
    'test_size_alias': TEST_SIZE,   # só informativo, para manter consistência com RF básico
    'random_state': RT_RANDOM_STATE,
    'acc_train': acc_train,
    'acc_test':  acc_test,
    'f1_train':  f1_tr,
    'f1_test':   f1_ts,
    'precision_train': prec_tr,
    'precision_test':  prec_ts,
    'recall_train':    rec_tr,
    'recall_test':     rec_ts,
    'auc_train': auc_tr,
    'auc_test':  auc_ts
}])
metrics_basic.to_csv(BASICO_CSV / f'xgb_model_basic.csv', index=False)

# === Adiciona metrics_basic ao final do PDF resumo (tenta mesclar) ===
metrics_pdf_path = BASICO_DIR / f'xgb_basic_metrics_page_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.pdf'
main_pdf   = PDF_OUT / f'XGB_CV_SUMMARY_{N_SPLITS}.pdf'
merged_pdf = PDF_OUT / f'XGB_CV_SUMMARY_{N_SPLITS}_merged.pdf'
try:
    fig, ax = plt.subplots(figsize=(11.69, 8.27))  # A4 landscape
    ax.axis('off')
    display_df = metrics_basic.copy().round(4)
    table = ax.table(cellText=display_df.values, colLabels=display_df.columns, loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.2)
    ax.set_title('Tabela: Métricas - XGBoost Básico (9/1 via KFold)', pad=20)
    plt.tight_layout()
    with PdfPages(metrics_pdf_path) as pdf2:
        pdf2.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    try:
        from PyPDF2 import PdfMerger
        merger = PdfMerger()
        if main_pdf.exists():
            merger.append(str(main_pdf))
        merger.append(str(metrics_pdf_path))
        with open(merged_pdf, 'wb') as f:
            merger.write(f)
        merged_pdf.replace(main_pdf)
        if metrics_pdf_path.exists():
            metrics_pdf_path.unlink()
        print('metrics_basic (XGB 9/1) adicionada ao final do PDF resumo.')
    except Exception as e_merge:
        print('Não foi possível mesclar PDFs automaticamente (PyPDF2 pode estar ausente).')
        print('O PDF com as métricas foi salvo em:', metrics_pdf_path, 'Erro:', e_merge)
except Exception as e:
    print('Não foi possível gerar a página PDF de metrics_basic:', e)

# Dataset augmentado (original com y, y_pred, y_proba) para salvar na pasta basico e na cv CSV
try:
    X_test_reset = X_test.reset_index().rename(columns={'index': 'orig_index'})
    preds_df = pd.DataFrame({
        'orig_index': X_test_reset['orig_index'],
        'y_pred': y_pred_test_bin,
        'y_proba': y_proba_test
    })
    df_orig = df.reset_index().rename(columns={'index': 'orig_index'})
    df_augmented = pd.merge(df_orig, preds_df, how='left', on='orig_index')
    df_augmented.to_csv(BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv', index=False)
    df_augmented.to_csv(CSV_OUT     / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv', index=False)
except Exception as e_aug:
    print('Não foi possível gerar dataset augmentado do basico:', e_aug)

# Seleciona exemplos aleatórios do X_test para explicação (12 por padrão)
num_instancias = 12
np.random.seed(42)
indices_aleatorios = np.random.choice(X_test.index, size=min(num_instancias, len(X_test)), replace=False)

print('Modelo salvo em:', model_name)
print('Métricas salvas em:', BASICO_CSV / f'xgb_model_basic.csv')
print('Dataset augmentado salvo em:', BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv')
print('Índices amostra aleatorios uso diverso:', indices_aleatorios)


In [ ]:
# Salvar X_test_basic_full.csv e X_train_basic_full.csv
# com todas as colunas do arquivo cru + y, y_pred, y_proba
import pandas as pd

# Carregar o arquivo cru
raw_df = pd.read_csv(DATASET_PATH)

# Garante a coluna 'y' no raw_df (sem amarrar só em 'diagnosis')
if 'y' not in raw_df.columns:
    if 'diagnosis' in raw_df.columns:
        # Ex.: WDBC (M/B -> 1/0)
        raw_df['y'] = raw_df['diagnosis'].map({'M': 1, 'B': 0})
    else:
        raise RuntimeError("Coluna 'y' não encontrada em raw_df e não há lógica específica para criar 'y' (ajuste conforme o dataset).")

# Função para montar e salvar o DataFrame completo (alinhando pelos índices originais)
def save_full_split(indices, y_true, y_pred, y_proba, split_name):
    idx = pd.Index(indices)
    df_full = raw_df.loc[idx].copy()

    y_true_s  = pd.Series(y_true,  index=idx)
    y_pred_s  = pd.Series(y_pred,  index=idx)
    y_proba_s = pd.Series(y_proba, index=idx)

    df_full['y']       = y_true_s.values
    df_full['y_pred']  = y_pred_s.values
    df_full['y_proba'] = y_proba_s.values

    df_full.to_csv(BASICO_CSV / f'X_{split_name}_basic_full.csv', index=True)

# Usa os índices originais já preservados em X_train / X_test (9/1 via KFold)
save_full_split(X_test.index,  y_test,  y_pred_test_bin,  y_proba_test,  'test')
save_full_split(X_train.index, y_train, y_pred_train_bin, y_proba_train, 'train')
